In [ ]:
#| default_exp visualization


# Classification Interpretation

> Relate the latent classification axis to interpretable image feature scores.

After **`train_xaaenet`**, compare the model’s latent codes to interpretable pixel scores: latent matrix `z` (one row per image) and a feature table from `compute_feature_score_table`, on the **same images in the same order**. For **binary** classification, encode one level as class **A** (`target_score = 1.0`) and the other as class **B** (`0.0`).

Call **`run_pls_feature_figures`** to build, display, and optionally save two figures:

1. **Alignment panels** — one scatter plot per feature (latent axis vs feature value);
2. **Importance ranking** — bars ranked by how strongly each feature follows that axis.

A full end-to-end example lives in the tutorial notebook.

## `run_pls_feature_figures`

```python
from tell_me_why.feature_scores import compute_feature_score_table
from tell_me_why.visualization import run_pls_feature_figures

# z: (n_images, latent_dim) from your trained xAAEnet on the split you analyse
# target_score: 1.0 = class A, 0.0 = class B, same row order as z
df_features = compute_feature_score_table([str(p) for p in image_paths])

out = run_pls_feature_figures(
    z,
    target_score,
    df_features,
    target_label="A",
    mask_positive=target_score.astype(bool),  # True for class A
    positive_label="A",
    negative_label="B",
    save_dir="results/pls",
    show=True,
)

fig_panels = out["alignment_fig"]
fig_ranking = out["importance_fig"]
ranked_features = out["importance_rank"]
```

### Arguments

| Argument | Role |
|----------|------|
| `z` | Latent codes from your trained model: one row per image, shape `(number of images, latent dimension)` — e.g. 500 images × 128-D latent space → `(500, 128)` |
| `target_score` | Binary targets: `1.0` = class A, `0.0` = class B |
| `feature_table` | DataFrame from `compute_feature_score_table` |
| `target_label` | Name of class A on the importance chart |
| `mask_positive` | Boolean mask where `target_score == 1.0` (class A) |
| `positive_label`, `negative_label` | Legend labels for class A and class B on alignment panels |
| `feature_columns` | Columns to plot (default: all eleven feature scores) |
| `save_dir` | Saves `feature_alignment_panels.png` and `feature_importance_ranking.png` |
| `show` | Call `plt.show()` for each figure (`False` keeps figures only in `out`) |


## Reading the alignment panels

Each panel plots **PLS component 1** (horizontal, latent direction linked to your classes) against one **standardized feature score** on the vertical axis (each column from `compute_feature_score_table` is scaled to mean 0 and standard deviation 1 over your sample, so brightness, texture, etc. are comparable on the same plot grid).

![Feature alignment panels — binary example (class A vs class B)](images/feature_alignment_panels_example.png)

| Element | Meaning |
|---------|--------|
| Grey / blue points | Class B / class A |
| Green line | Linear trend across all points; legend shows Pearson `r` |
| Top-left | Strength of association (`|r²|`) and regression p-value |
| ▲ / ▼ signed r² | Feature increases or decreases when PLS1 moves toward class A (`target_label`) |

### Two extremes cases

**1. Horizontal green line — weak link to classification**

The regression line is flat and the cloud does not climb or fall along PLS1. The feature co-varies little with this latent axis: the model is probably **not** using this cue to distinguish the two classes (e.g. *skewness*, *symmetry* in the example above).

**2. Diagonal green line — captured bias**

Points spread along a clear slope: low PLS1 ↔ low feature value on one side, high PLS1 ↔ high value on the other. Class A and class B often separate left–right **and** bottom–top. That is a strong sign that the model may **rely on this pixel-level cue** to classify (e.g. *redness_dominance*, *brightness* in the example figure, with class A = Male and class B = Female).

Use the panels to **confirm** features that stand out in the importance ranking, not to rank them (that is the bar chart’s job).


## Reading the importance ranking

One horizontal bar per feature, sorted by **|signed r²|** with PLS component 1 (strongest at the top).

![Feature importance ranking — binary example (class A vs class B)](images/feature_importance_ranking_example.png)

| Element | Meaning |
|---------|--------|
| Green bar (right) | Feature increases with PLS1 toward class A (`target_label`) |
| Red bar (left) | Feature decreases when PLS1 moves toward class A (anti-aligned → class B) |
| Bar length | Strength of linear association (not a p-value) |
| Near-zero bar | Little linear coupling to the latent axis on the full sample |

### How to read the chart

- **Top of the list:** Features most aligned with the latent decision axis — start here when asking what the model might use in image space.
- **Long green bars:** Cues that rise with PLS1 toward class A; open the matching alignment panel — you should see a **diagonal** green line and separated classes.
- **Short or near-zero bars:** Weak linear link to the latent axis; the corresponding alignment panel usually shows a **horizontal** green line and a mixed cloud.
- **Compare with alignment panels:** The ranking is a compact summary; the grid validates whether a large bar reflects a real visual bias or an outlier-driven slope.

### Reading the example figure

On this human gender classification (class A = Male, class B = Female), the chart suggests the latent classification axis is **mostly carried by color and luminance**, not by shape regularity:

- **Strong cues:** *redness_dominance* and *brightness* have the longest green bars (high signed r²). The model’s PLS1 direction co-varies strongly with “more red” and “brighter” toward class A — plausible pixel-level biases for this task.
- **Weak cues:** *symmetry_error*, *skewness*, *variance*, and *eye_region_contrast* have bars close to zero. Along PLS1, facial symmetry (and these other scores) show **little linear association** with the classification axis in this sample — not evidence that the model relied on them here.

This is a **hypothesis from alignment statistics**, not a proof of what the network computes internally. Always cross-check the alignment panels for the top features.

Use this figure for reporting and comparing runs; use the alignment panels for qualitative checks on the top features.


In [ ]:
#| export
from __future__ import annotations

from collections.abc import Sequence
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler

DEFAULT_FEATURE_SCORE_COLUMNS: tuple[str, ...] = (
    "brightness",
    "variance",
    "skewness",
    "color_covariance_red_blue",
    "redness_dominance",
    "symmetry_error",
    "top_bottom_brightness_ratio",
    "center_texture",
    "eye_region_contrast",
    "jaw_texture",
    "fft_high_frequency_ratio",
)


In [ ]:
#| exporti
def pearson_r2_signed(x: np.ndarray, y: np.ndarray) -> float:
    r, _ = stats.pearsonr(np.asarray(x).ravel(), np.asarray(y).ravel())
    return float(np.sign(r) * r**2)


def supervised_pls_latent(
    z: np.ndarray,
    target_score: np.ndarray,
    *,
    n_components: int = 2,
) -> tuple[np.ndarray, PLSRegression, float, float]:
    z = np.asarray(z, dtype=np.float64)
    y = np.asarray(target_score, dtype=np.float64).reshape(-1, 1)
    pls = PLSRegression(n_components=n_components)
    pls.fit(z, y)
    z_pls = pls.transform(z)
    r2_c1 = stats.pearsonr(z_pls[:, 0], y.ravel())[0] ** 2
    r2_c2 = stats.pearsonr(z_pls[:, 1], y.ravel())[0] ** 2 if n_components > 1 else float("nan")
    return z_pls, pls, float(r2_c1), float(r2_c2)


def compute_pls_feature_alignment(
    pls_c1: np.ndarray,
    pls_c2: np.ndarray,
    features_z: np.ndarray,
    feature_names: Sequence[str],
) -> dict[str, dict[str, Any]]:
    pls_c1 = np.asarray(pls_c1).ravel()
    pls_c2 = np.asarray(pls_c2).ravel()
    out: dict[str, dict[str, Any]] = {}
    for j, name in enumerate(feature_names):
        vals = features_z[:, j]
        r_x, _ = stats.pearsonr(pls_c1, vals)
        r_y, _ = stats.pearsonr(pls_c2, vals)
        _, _, r_reg, p_reg, _ = stats.linregress(pls_c1, vals)
        out[name] = {
            "vector_xy": (float(r_x), float(r_y)),
            "r2_signed": pearson_r2_signed(pls_c1, vals),
            "p_value": float(p_reg),
            "linregress_r": float(r_reg),
        }
    return out


def _standardize_features(features: np.ndarray) -> np.ndarray:
    return StandardScaler().fit_transform(np.asarray(features, dtype=np.float64))


In [ ]:
#| exporti
def plot_feature_alignment_panels(
    pls_c1: np.ndarray,
    features_z: np.ndarray,
    feature_names: Sequence[str],
    alignment: dict[str, dict[str, Any]],
    *,
    mask_positive: np.ndarray | None = None,
    positive_label: str = "positive",
    negative_label: str = "negative",
    cols: int = 3,
    title: str | None = None,
    show: bool = True,
) -> plt.Figure:
    """Grid of scatter plots: PLS component 1 vs each standardized feature score."""
    pls_c1 = np.asarray(pls_c1).ravel()
    n_feat = len(feature_names)
    rows = int(np.ceil(n_feat / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4.5))
    fig.patch.set_facecolor("#0e1117")
    axes_flat = np.atleast_1d(axes).flatten()

    if mask_positive is None:
        mask_positive = np.ones(len(pls_c1), dtype=bool)
    mask_negative = ~mask_positive

    def pval_str(p: float) -> str:
        if p < 0.001:
            return "p < 0.001"
        if p < 0.01:
            return "p < 0.01"
        if p < 0.05:
            return "p < 0.05"
        return f"p = {p:.3f}"

    for i, feat in enumerate(feature_names):
        ax = axes_flat[i]
        ax.set_facecolor("#0d1117")
        vals = features_z[:, i]
        ax.scatter(
            pls_c1[mask_negative],
            vals[mask_negative],
            c="#444c56",
            s=6,
            alpha=0.4,
            linewidths=0,
            label=negative_label,
            zorder=1,
        )
        ax.scatter(
            pls_c1[mask_positive],
            vals[mask_positive],
            c="#63b3ed",
            s=8,
            alpha=0.6,
            linewidths=0,
            label=positive_label,
            zorder=2,
        )
        xr = np.linspace(pls_c1.min(), pls_c1.max(), 200)
        m, b, r_reg, p_reg, _ = stats.linregress(pls_c1, vals)
        ax.plot(xr, m * xr + b, color="#56de91", lw=2.0, alpha=0.9, zorder=3, label=f"r = {r_reg:.2f} (all)")

        r2_signed = alignment[feat]["r2_signed"]
        r2_col = "#56de91" if r2_signed >= 0 else "#fc6e6e"
        r2_sign = "▲" if r2_signed >= 0 else "▼"
        ax.set_title(feat, fontsize=11, fontweight="bold", pad=6, color="#c9d1d9")
        ax.set_xlabel("PLS component 1", fontsize=8, color="#8b949e")
        ax.set_ylabel(feat, fontsize=8, color="#8b949e")
        ax.text(
            0.03,
            0.97,
            f"|r²| = {abs(r2_signed):.3f}  |  {pval_str(p_reg)}",
            transform=ax.transAxes,
            fontsize=8,
            va="top",
            ha="left",
            color="#c9d1d9",
            bbox=dict(boxstyle="round,pad=0.35", fc="#1c2128", ec="#30363d", alpha=0.85),
        )
        ax.text(
            0.97,
            0.05,
            f"{r2_sign} signed r² = {r2_signed:+.3f}",
            transform=ax.transAxes,
            fontsize=8.5,
            fontweight="bold",
            va="bottom",
            ha="right",
            color=r2_col,
        )
        ax.legend(loc="lower right", fontsize=7, framealpha=0.3)
        ax.grid(True, alpha=0.12)
        ax.tick_params(colors="#8b949e")
        for spine in ax.spines.values():
            spine.set_edgecolor("#30363d")

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(
        title or "Feature alignment vs PLS component 1\n(▲ positive signed r²  |  ▼ negative)",
        y=1.01,
        fontsize=12,
        fontweight="bold",
        color="#c9d1d9",
    )
    fig.tight_layout()
    if show:
        plt.show()
    return fig


In [ ]:
#| exporti
def plot_feature_importance_ranking(
    importance_rank: Sequence[str],
    alignment: dict[str, dict[str, Any]],
    *,
    target_label: str = "target",
    show: bool = True,
) -> plt.Figure:
    """Horizontal bar chart of signed r² vs PLS1, sorted by |r²|."""
    import matplotlib.patches as mpatches

    r2_vals = np.array([alignment[f]["r2_signed"] for f in importance_rank])
    colors = ["#56de91" if v >= 0 else "#fc6e6e" for v in r2_vals]
    fig, ax = plt.subplots(figsize=(11, 5))
    fig.patch.set_facecolor("#0e1117")
    ax.set_facecolor("#0d1117")
    ax.barh(list(importance_rank), r2_vals, color=colors, height=0.55, edgecolor="#21262d", linewidth=0.6)
    ax.axvline(0, color="#8b949e", lw=1.2)
    ax.set_xlabel(
        f"Signed r² (full sample)  →  positive = increases with PLS 1 toward '{target_label}'",
        fontsize=10,
        color="#c9d1d9",
    )
    ax.set_title(
        f"Feature importance ranking — PLS axis '{target_label}'",
        fontweight="bold",
        pad=10,
        loc="left",
        color="white",
    )
    ax.tick_params(colors="#8b949e")
    for spine in ax.spines.values():
        spine.set_edgecolor("#30363d")
    ax.grid(True, axis="x", alpha=0.15)
    ax.legend(
        handles=[
            mpatches.Patch(color="#56de91", label=f"Aligned with PLS 1 (→ {target_label})"),
            mpatches.Patch(color="#fc6e6e", label="Anti-aligned with PLS 1"),
        ],
        loc="lower right",
        framealpha=0.3,
    )
    fig.tight_layout()
    if show:
        plt.show()
    return fig


In [ ]:
#| export
def run_pls_feature_figures(
    z: np.ndarray,
    target_score: np.ndarray,
    feature_table,
    *,
    feature_columns: Sequence[str] | None = None,
    target_label: str = "target",
    mask_positive: np.ndarray | None = None,
    positive_label: str = "positive",
    negative_label: str = "negative",
    save_dir: str | Path | None = None,
    show: bool = True,
) -> dict[str, Any]:
    """Build alignment panels and importance ranking; display and optionally save PNGs."""
    feature_columns = list(feature_columns or DEFAULT_FEATURE_SCORE_COLUMNS)
    features_z = _standardize_features(feature_table[feature_columns].values)
    z_pls, pls, r2_c1, r2_c2 = supervised_pls_latent(z, target_score)
    pls_c1, pls_c2 = z_pls[:, 0], z_pls[:, 1]
    alignment = compute_pls_feature_alignment(pls_c1, pls_c2, features_z, feature_columns)
    importance_rank = sorted(feature_columns, key=lambda f: abs(alignment[f]["r2_signed"]), reverse=True)
    alignment_fig = plot_feature_alignment_panels(
        pls_c1, features_z, feature_columns, alignment,
        mask_positive=mask_positive, positive_label=positive_label, negative_label=negative_label, show=show,
    )
    importance_fig = plot_feature_importance_ranking(importance_rank, alignment, target_label=target_label, show=show)
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        alignment_fig.savefig(save_dir / "feature_alignment_panels.png", dpi=150, bbox_inches="tight", facecolor=alignment_fig.get_facecolor())
        importance_fig.savefig(save_dir / "feature_importance_ranking.png", dpi=150, bbox_inches="tight", facecolor=importance_fig.get_facecolor())
    return {
        "alignment_fig": alignment_fig,
        "importance_fig": importance_fig,
        "importance_rank": importance_rank,
        "alignment": alignment,
    }
